# Diffraction reflection angles + 3D view (Bernina, `you` geometry)

Compute diffractometer angles for a given `(h,k,l)` reflection using diffcalc's
`you` UB geometry — note this beamline's convention calls the out-of-plane
detector circle **gamma**, where diffcalc itself calls it `nu`; that rename
happens automatically everywhere below.

Fix any subset of the constraint angles (checkbox) and diffcalc solves for the
rest. Unfixed reflections may have more than one valid solution branch, shown
as columns `sol. 0`, `sol. 1`, ....

**Click a solution column** to drive the 3D view; it highlights in light blue
and the mechanism moves to those angles. **zero** returns every circle to 0,
which is also the state the view starts in.

The 3D panel's own controls sit under the plot by default (uncheck *controls
under the plot* to put them beside it). Its options:

- each circle's arc shows *how far it has turned*; at 0 it falls back to the
  short arrow indicating the positive direction. A negative angle is drawn as
  the complementary 2π − |angle| arc, so the arrowhead never flips.
- **show rotation circles** hides the whole ring/arc set.
- **show angle controls** brings back the per-circle sliders (off here, since
  the solution table drives them).
- **show reciprocal vectors** draws k (ending at the sample), k&prime; (towards
  the detector, same length) and Q = k&prime; &minus; k from the sample, all
  labelled, with thin lines closing the quadrilateral.
- **show reciprocal unit cell** draws labelled a\*, b\*, c\* from the sample in
  red, green and blue, body completed in thin black lines. It rotates with
  mu/eta/chi/phi, so the tip of Q lands on the cell node of the reflection.
- **label Q with hkl + cell around it** appends the reflection indices to the Q
  label and outlines the cell that point sits in — hkl (0.1, 2.99, 3) gives the
  cell spanning h 0–1, k 2–3, l 3–4.
- **show Ewald sphere** draws the sphere of radius |k| centred on the tail of k,
  translucent blue, clipped to the view box (the view never rescales to it).
- **show detector cut of the Ewald sphere** shades in red the patch that the
  detector actually sees, from its distance and its width/height below.

Reciprocal lengths follow diffcalc's own 2π convention (|k| = 2π/λ, |Q| = 2π/d,
|a\*| = 2π/a); one slider sets how many cm of the direct-space scene represent
1 Å⁻¹ (5 cm by default).


In [1]:
import html
import json
import sys
from pathlib import Path

try:
    import bernina_prep  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parents[1]))

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from bernina_prep.diffraction import (
    build_ubcalc,
    k_magnitude,
    make_constraints,
    reciprocal_basis,
    solution_angles,
    solve_hkl,
)

UC_W = "150px"
ANG_W = "150px"
HKL_W = "150px"
DESC_W = "58px"

LAT_NAME = "custom lattice"

# --- controls ---------------------------------------------------------------
# Built explicitly and wired by name; fixed px everywhere (a % width can
# collapse the whole column to nothing under voila) and core widgets only.

ctrl = {
    "uc_a": widgets.FloatText(description="a (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_b": widgets.FloatText(description="b (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_c": widgets.FloatText(description="c (uc)", value=5.43, layout=widgets.Layout(width=UC_W)),
    "uc_alpha": widgets.FloatText(description="alpha", value=90, layout=widgets.Layout(width=UC_W)),
    "uc_beta": widgets.FloatText(description="beta", value=90, layout=widgets.Layout(width=UC_W)),
    "uc_gamma": widgets.FloatText(description="gamma", value=90, layout=widgets.Layout(width=UC_W)),
    "h": widgets.IntText(value=1, description="h", layout=widgets.Layout(width=HKL_W),
                         style={"description_width": DESC_W}),
    "k": widgets.IntText(value=1, description="k", layout=widgets.Layout(width=HKL_W),
                         style={"description_width": DESC_W}),
    "l": widgets.IntText(value=1, description="l", layout=widgets.Layout(width=HKL_W),
                         style={"description_width": DESC_W}),
    "surface_normal": widgets.Text(value="(1,1,1)", description="surf. normal",
                                   layout=widgets.Layout(width="340px"),
                                   style={"description_width": "90px"}),
    "orientations": widgets.Textarea(
        value="(1,1,1) > (0,0,1)\n(1,1,-1) > (1,0,0)",
        description="orientations",
        placeholder='one "hkl > xyz" pair per line',
        layout=widgets.Layout(width="340px", height="60px"),
        style={"description_width": "90px"},
    ),
}

angle_names = ["delta", "gam", "qaz", "naz", "alpha", "beta", "psi", "a_eq_b",
               "mu", "eta", "chi", "phi", "mu_is_gam"]
default_fixed = {"eta", "chi", "phi"}
for name in angle_names:
    if name in ("a_eq_b", "mu_is_gam"):
        ctrl[name] = widgets.Checkbox(value=False, indent=False,
                                      layout=widgets.Layout(width=ANG_W))
    else:
        ctrl[name] = widgets.FloatText(value=0, layout=widgets.Layout(width=ANG_W))
    ctrl[f"{name}_fixed"] = widgets.Checkbox(value=name in default_fixed,
                                             description="fix", indent=False,
                                             layout=widgets.Layout(width="70px"))

ctrl["energy"] = widgets.FloatSlider(value=8000, min=1000, max=20000, step=10,
                                     readout=False, continuous_update=False,
                                     layout=widgets.Layout(width="170px"))
energy_value = widgets.BoundedFloatText(description="energy", value=8000, min=1000,
                                        max=20000, step=10,
                                        layout=widgets.Layout(width="150px"),
                                        style={"description_width": DESC_W})
widgets.link((energy_value, "value"), (ctrl["energy"], "value"))
energy_row = widgets.HBox([energy_value, widgets.Label("eV"), ctrl["energy"]])

ctr_uc = widgets.HBox([ctrl[n] for n in ("uc_a", "uc_b", "uc_c", "uc_alpha", "uc_beta", "uc_gamma")],
                      layout=widgets.Layout(flex_flow="row wrap"))
ctr_hkl = widgets.HBox([ctrl["h"], ctrl["k"], ctrl["l"]],
                       layout=widgets.Layout(flex_flow="row wrap"))
ctr_or = widgets.VBox([ctrl["surface_normal"], ctrl["orientations"]])
ctr_ang = widgets.VBox([
    widgets.HBox([widgets.Label(name, layout=widgets.Layout(width="90px")),
                  ctrl[name], ctrl[f"{name}_fixed"]])
    for name in angle_names
])
controls = widgets.VBox([ctr_uc, ctr_hkl, ctr_or, ctr_ang, energy_row],
                        layout=widgets.Layout(width="380px", flex="0 0 auto"))

# --- solutions table, with one clickable column per solution branch ---------
# Rendered as HTML (compact, and the numbers stay aligned) with a row of
# buttons acting as the column headers, so a click can actually reach python -
# a pandas DataFrame's own row highlight is browser text selection and cannot.

ROW_LABEL_W = 78
COL_W = 96
SELECTED_BG = "#cfe8fb"   # light blue

state = {"df": None, "selected": None, "ubcalc": None, "cons": None}

solution_buttons = widgets.HBox([])
table_html = widgets.HTML()
solve_msg = widgets.HTML()
output = widgets.Output()   # kept for the cells below


def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "NaN"
    return f"{float(v):.3f}"


def render_table():
    df = state["df"]
    if df is None:
        table_html.value = ""
        return
    sel = state["selected"]
    head = "".join(
        f'<th style="width:{COL_W}px;text-align:right;padding:2px 6px;'
        f'border-bottom:1px solid #ccc;'
        f'{"background:" + SELECTED_BG + ";" if i == sel else ""}">{html.escape(str(c))}</th>'
        for i, c in enumerate(df.columns)
    )
    body = []
    for name, row in df.iterrows():
        cells = "".join(
            f'<td style="width:{COL_W}px;text-align:right;padding:1px 6px;'
            f'font-variant-numeric:tabular-nums;'
            f'{"background:" + SELECTED_BG + ";" if i == sel else ""}">{_fmt(row[c])}</td>'
            for i, c in enumerate(df.columns)
        )
        body.append(
            f'<tr><td style="width:{ROW_LABEL_W}px;text-align:right;padding:1px 6px;'
            f'font-weight:600;">{html.escape(str(name))}</td>{cells}</tr>'
        )
    table_html.value = (
        '<table style="border-collapse:collapse;font-size:12px;">'
        f'<tr><td style="width:{ROW_LABEL_W}px"></td>{head}</tr>'
        + "".join(body)
        + "</table>"
    )


def select_solution(index):
    """Pick a solution branch (or None for 'all circles at zero')."""
    state["selected"] = index
    # children[0] is the "zero" button, so the solution buttons start at 1
    for i, b in enumerate(solution_buttons.children[1:]):
        b.button_style = "info" if i == index else ""
    solution_buttons.children[0].button_style = "" if index is not None else "info"
    render_table()
    push_view()


def rebuild_solution_buttons():
    df = state["df"]
    if df is None:
        solution_buttons.children = ()
        return
    buttons = []
    for i, col in enumerate(df.columns):
        b = widgets.Button(description=str(col), tooltip=f"show {col} in the 3D view",
                           layout=widgets.Layout(width=f"{COL_W}px", margin="0"))
        b.on_click(lambda _b, idx=i: select_solution(idx))
        buttons.append(b)
    zero = widgets.Button(description="zero", tooltip="move all circles back to 0",
                          layout=widgets.Layout(width=f"{ROW_LABEL_W}px", margin="0"))
    zero.on_click(lambda _b: select_solution(None))
    solution_buttons.children = [zero] + buttons


# --- the 3D view ------------------------------------------------------------
# The page is embedded as an iframe with srcdoc, which inherits this page's
# origin, so it can read the hidden payload element below. That is how a
# solution gets pushed into the scene without a custom widget.

_html_path = Path.cwd() / "diffractometer_3d.html"
if not _html_path.exists():
    _html_path = Path(__file__).parent / "diffractometer_3d.html"  # pragma: no cover
_scene_html = _html_path.read_text()

view_frame = widgets.HTML(
    f'<iframe id="diff3d" srcdoc="{html.escape(_scene_html)}" '
    'style="width:100%; height:760px; border:1px solid #ddd;"></iframe>',
    layout=widgets.Layout(width="100%"),
)
payload_holder = widgets.HTML()
_push_count = {"n": 0}


def push_view():
    """Write the current solution into the element the 3D page polls."""
    angles = {n: 0.0 for n in ("mu", "eta", "chi", "phi", "delta", "gamma")}
    label = ""
    if state["df"] is not None and state["selected"] is not None:
        angles = solution_angles(state["df"], state["selected"])
        label = str(state["df"].columns[state["selected"]])
    msg = {
        "angles": angles,
        "kMag": k_magnitude(ctrl["energy"].value),
        "label": label,
        # the page ignores repeats, so make every push a distinct string
        "n": _push_count["n"],
    }
    if state["ubcalc"] is not None:
        msg["ub"] = reciprocal_basis(state["ubcalc"])
    # lets the view label Q with its hkl and outline the cell it sits in
    msg["hkl"] = [ctrl["h"].value, ctrl["k"].value, ctrl["l"].value]
    _push_count["n"] += 1
    payload_holder.value = (
        '<span id="diff3d-payload" style="display:none">'
        + html.escape(json.dumps(msg))
        + "</span>"
    )


# --- solve ------------------------------------------------------------------

def recompute(*_):
    global last_ubcalc, last_cons
    try:
        ubcalc = build_ubcalc(
            LAT_NAME,
            ctrl["uc_a"].value, ctrl["uc_b"].value, ctrl["uc_c"].value,
            ctrl["uc_alpha"].value, ctrl["uc_beta"].value, ctrl["uc_gamma"].value,
            [ln.strip() for ln in ctrl["orientations"].value.splitlines() if ln.strip()],
            ctrl["surface_normal"].value,
        )
        fixed = {n: ctrl[n].value for n in angle_names if ctrl[f"{n}_fixed"].value}
        cons = make_constraints(fixed)
        df = solve_hkl(ubcalc, cons, ctrl["h"].value, ctrl["k"].value,
                       ctrl["l"].value, ctrl["energy"].value)
    except Exception as exc:
        solve_msg.value = f'<span style="color:#b00">{html.escape(str(exc))}</span>'
        state["df"] = None
        rebuild_solution_buttons()
        render_table()
        push_view()
        return

    solve_msg.value = ""
    state.update(df=df, ubcalc=ubcalc, cons=cons)
    last_ubcalc, last_cons = ubcalc, cons        # used by the cells below
    if state["selected"] is not None and state["selected"] >= len(df.columns):
        state["selected"] = None
    rebuild_solution_buttons()
    select_solution(state["selected"])           # re-renders and re-pushes


for _w in ctrl.values():
    _w.observe(recompute, names="value")
recompute()

solutions_panel = widgets.VBox(
    [widgets.HTML("<b>solutions</b> &mdash; click a column to show it in the 3D view"),
     solve_msg, solution_buttons, table_html],
    layout=widgets.Layout(flex="0 0 auto", margin="0 0 0 12px"),
)
view_panel = widgets.VBox(
    [view_frame, payload_holder],
    layout=widgets.Layout(flex="1 1 520px", min_width="0", margin="0 0 0 12px"),
)

# Three columns side by side on a wide screen; wraps to a single scrollable
# column on a narrow one. Fixed basis on each - never a percentage.
display(widgets.HBox(
    [controls, solutions_panel, view_panel],
    layout=widgets.Layout(flex_flow="row wrap", width="100%", align_items="flex-start"),
))


## Table of reflections

Reuses the crystal/orientation and fixed constraints set above. Enter one
`h,k,l` per line and pick which solution branch to report when a reflection
has more than one; reflections diffcalc can't solve are flagged in the
`error` column rather than aborting the whole table.

In [2]:
from bernina_prep.diffraction import reflection_table

hkl_text = widgets.Textarea(
    value="1,0,0\n0,1,0\n0,0,1\n1,1,1",
    description="h,k,l list",
    layout=widgets.Layout(width="200px", height="120px"),
)
solution_idx = widgets.IntText(value=0, description="solution #")
compute_btn = widgets.Button(description="Compute table", button_style="primary")
table_out = widgets.Output()


def parse_hkl_lines(text):
    reflections = []
    for line in text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        h, k, l = (float(x) for x in line.replace(";", ",").split(","))
        reflections.append((h, k, l))
    return reflections


def on_compute(_):
    table_out.clear_output()
    with table_out:
        if "last_ubcalc" not in globals():
            print("Set up the crystal/orientation above first (run the single-reflection cell).")
            return
        reflections = parse_hkl_lines(hkl_text.value)
        display(reflection_table(last_ubcalc, last_cons, reflections, ctrl["energy"].value, solution=solution_idx.value))


compute_btn.on_click(on_compute)
display(widgets.VBox([widgets.HBox([hkl_text, solution_idx]), compute_btn, table_out]))


## Detector footprint (Q / hkl range)

For the currently-set crystal/orientation and a chosen `(h,k,l)`/solution
branch above, computes the range of reciprocal-lattice points a flat area
detector actually covers, given its distance from the sample (`tdet`) and
its physical size. Uses diffcalc's own `q_phi` convention directly (not a
second geometry library), extended from the single central ray to the
whole detector face, so it's consistent with the angles calculated above.

In [ ]:
from bernina_prep.diffraction import solve_hkl_positions, detector_hkl_grid, detector_hkl_summary

tdet_w = widgets.FloatText(value=500.0, description="tdet [mm]")
det_w_w = widgets.FloatText(value=77.0, description="width [mm]")
det_h_w = widgets.FloatText(value=77.0, description="height [mm]")
grid_n_w = widgets.IntText(value=9, description="grid n")
footprint_solution_idx = widgets.IntText(value=0, description="solution #")
footprint_btn = widgets.Button(description="Compute Q range", button_style="primary")
plot_btn = widgets.Button(description="Show 3D plot (slower)", button_style="")
footprint_out = widgets.Output()
plot_out = widgets.Output()

_last_footprint = {}


def _current_position():
    h, k, l = ctrl["h"].value, ctrl["k"].value, ctrl["l"].value
    energy = ctrl["energy"].value
    result = solve_hkl_positions(last_ubcalc, last_cons, h, k, l, energy)
    idx = footprint_solution_idx.value if footprint_solution_idx.value < len(result) else len(result) - 1
    pos, _ = result[idx]
    return pos, energy


def on_footprint(_):
    footprint_out.clear_output()
    with footprint_out:
        if "last_ubcalc" not in globals():
            print("Set up the crystal/orientation above first (run the single-reflection cell).")
            return
        pos, energy = _current_position()
        H, K, L = detector_hkl_grid(
            last_ubcalc, pos, energy, tdet_w.value, det_w_w.value, det_h_w.value, grid_n_w.value
        )
        _last_footprint.update(pos=pos, energy=energy, H=H, K=K, L=L)
        summary = detector_hkl_summary(
            last_ubcalc, pos, energy, tdet_w.value, det_w_w.value, det_h_w.value, grid_n_w.value
        )
        print(f"Detector center at delta={pos.delta:.3f}, gamma={pos.nu:.3f}, tdet={tdet_w.value:.1f} mm")
        display(summary.to_frame("value"))


def on_plot(_):
    plot_out.clear_output()
    with plot_out:
        if not _last_footprint:
            print("Compute the Q range first.")
            return
        import matplotlib.pyplot as plt

        H, K, L = _last_footprint["H"], _last_footprint["K"], _last_footprint["L"]
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(projection="3d")
        ax.plot_surface(H, K, L, alpha=0.6)
        ax.set_xlabel("h")
        ax.set_ylabel("k")
        ax.set_zlabel("l")
        n = H.shape[0] // 2
        cx, cy, cz = H[n, n], K[n, n], L[n, n]
        arrowlen = 0.3 * max(H.max() - H.min(), K.max() - K.min(), L.max() - L.min(), 1e-6)
        ax.quiver(cx, cy, cz, arrowlen, 0, 0, color="r")
        ax.quiver(cx, cy, cz, 0, arrowlen, 0, color="g")
        ax.quiver(cx, cy, cz, 0, 0, arrowlen, color="b")
        plt.show()


footprint_btn.on_click(on_footprint)
plot_btn.on_click(on_plot)
display(
    widgets.VBox(
        [
            widgets.HBox([tdet_w, det_w_w, det_h_w, grid_n_w, footprint_solution_idx]),
            widgets.HBox([footprint_btn, plot_btn]),
            footprint_out,
            plot_out,
        ]
    )
)
